In [ ]:
# Ячейка 1: Импорты и загрузка
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
import matplotlib.pyplot as plt
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

feature_sets = lab_utils.load_feature_sets()
best_models = lab_utils.load_lab03_results()
print("Best models from L03:", best_models)

In [ ]:
# Ячейка 2: Обучение и калибровка
datasets = {
    'cardiovascular_risk': (cardio.drop('target',axis=1), cardio['target']),
    'credit_risk': (credit.drop('default',axis=1), credit['default'])
}

results = []

for ds_name, (X, y) in datasets.items():
    # Находим конфигурацию для датасета
    config = [m for m in best_models if m['dataset'] == ds_name][0]
    features = feature_sets[ds_name][config['feature_set']]
    
    # Split
    X_temp, X_test, y_temp, y_test = train_test_split(X[features], y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    
    # Базовая модель
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_s, y_train)
    
    # Калибровка
    calib_sigmoid = CalibratedClassifierCV(LogisticRegression(max_iter=1000), method='sigmoid', cv=3)
    calib_sigmoid.fit(X_train_s, y_train)
    
    calib_isotonic = CalibratedClassifierCV(LogisticRegression(max_iter=1000), method='isotonic', cv=3)
    calib_isotonic.fit(X_train_s, y_train)
    
    # Оценка на validation
    for variant, mdl in [
        ('uncalibrated', model),
        ('calibrated_sigmoid', calib_sigmoid),
        ('calibrated_isotonic', calib_isotonic)
    ]:
        y_prob = mdl.predict_proba(X_val_s)[:, 1]
        y_pred = mdl.predict(X_val_s)
        
        # ECE (Expected Calibration Error)
        n_bins = 10
        bins = np.linspace(0, 1, n_bins+1)
        ece = 0
        for i in range(n_bins):
            mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
            if mask.sum() > 0:
                ece += mask.sum() * abs(y_prob[mask].mean() - y_val[mask].mean())
        ece /= len(y_val)
        
        results.append({
            'dataset': ds_name,
            'model': 'LogisticRegression',
            'variant': variant,
            'split': 'validation',
            'brier': brier_score_loss(y_val, y_prob),
            'log_loss': log_loss(y_val, y_prob),
            'roc_auc': roc_auc_score(y_val, y_prob),
            'pr_auc': 0,
            'ece': ece
        })
        
        # График калибровки
        if variant != 'uncalibrated':
            plt.figure()
            plt.hist(y_prob, bins=20, alpha=0.5, label=variant)
            plt.xlabel('Predicted probability')
            plt.ylabel('Count')
            plt.title(f'{ds_name} - {variant}')
            plt.legend()
            plt.show()

calib_audit = pd.DataFrame(results)
calib_audit.to_csv('outputs/calibration_audit.csv', index=False)
print("Calibration audit:")
print(calib_audit[['dataset','variant','brier','log_loss','roc_auc','ece']])

In [ ]:
# Ячейка 3: Выбор лучшей калибровки
print("Выбор калибровки:")
for ds_name in calib_audit['dataset'].unique():
    subset = calib_audit[calib_audit['dataset'] == ds_name]
    best = subset.loc[subset['brier'].idxmin()]
    print(f"{ds_name}: {best['variant']} (brier={best['brier']:.3f})")